# X03 IOI Circuit


## Goal

Pushes circuit analysis from toy settings into a concrete natural-language behavior, making it a key bridge from paper diagrams to real tasks.


## Why this paper matters now

After the Anthropic core path, this forces you to handle messier behavior definitions, more heads, and a longer evidence chain.


## Notebook and deliverable

- Source: https://arxiv.org/abs/2211.00593
- Notebook: `notebooks/extensions/en/x03_ioi_circuit.ipynb`
- Prereqs: X02, M06
- Reproduce a teaching-scale IOI evidence chain in Colab, then write one brief stating which uncertainties appear here that were absent in the M06 toy trace.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import numpy as np

roles = np.array([
    [1.0, 0.0],  # subject = Alice
    [0.0, 1.0],  # indirect object = Bob
])
name_labels = ["Alice", "Bob"]
duplicate_head = np.array([[0.0, 0.2], [0.1, 0.0]])
name_mover = np.array([[0.2, 0.0], [0.0, 1.3]])
subject_suppressor = np.array([[1.0, 0.0], [0.0, 0.1]])

baseline_scores = np.array([0.5, 0.5])
full_scores = baseline_scores + roles[1] @ name_mover - roles[0] @ subject_suppressor + roles[1] @ duplicate_head
ablations = {
    "full": full_scores,
    "without name mover": baseline_scores - roles[0] @ subject_suppressor + roles[1] @ duplicate_head,
    "without subject suppressor": baseline_scores + roles[1] @ name_mover + roles[1] @ duplicate_head,
    "without duplicate head": baseline_scores + roles[1] @ name_mover - roles[0] @ subject_suppressor,
}

fig, ax = plt.subplots(figsize=(9, 4))
positions = np.arange(len(ablations))
width = 0.35
for idx, name in enumerate(name_labels):
    ax.bar(
        positions + (idx - 0.5) * width,
        [scores[idx] for scores in ablations.values()],
        width=width,
        label=name,
    )
ax.set_xticks(positions, list(ablations.keys()), rotation=15)
ax.set_ylabel("final logit score")
ax.set_title("Teaching-scale IOI evidence chain")
ax.legend()
plt.tight_layout()

for label, scores in ablations.items():
    winner = name_labels[int(np.argmax(scores))]
    print(label, "->", winner, np.round(scores, 3))


## Takeaway

Behavior-level circuit claims require ablations, not only one pretty path diagram.


## Colab-first replication workflow


### Before you run

- Write one prediction about the mechanism before you run the cells.
- Name the baseline you are comparing against.
- Decide what result would count as a failure of your favorite story.


### After you run

- Separate observation from inference in your notes.
- Mark one ambiguity that still remains after the reproduction.
- Write one next experiment that would reduce that ambiguity.


### Ship these outputs

- Reproduce a teaching-scale IOI evidence chain in Colab, then write one brief stating which uncertainties appear here that were absent in the M06 toy trace.
- One experiment log with the exact settings you changed.
- One paragraph called 'what this reproduction still does not prove'.


## Self-check questions

- Why is an IOI-style behavior task harder than a toy trace when you try to close an evidence loop?
- In your teaching-scale evidence chain, which edge looks most like 'necessary but not sufficient' evidence?
- If one head glows brightly in a graph, what extra evidence would you still need before claiming it drives the behavior?
- If you cannot answer at least two questions without reopening the notebook, rerun the reproduction and rewrite the memo.
